# Arousal Prediction v26

Major changes from v18:
1. **Physiologically-aware features**: pNN50, SDNN, LF/HF proxy from IBI; EDA phasic/tonic split; BVP-based HRV; first derivatives.
2. **Asymmetric windows for EDA/HR** (response latency: EDA peaks 1–4s after stimulus, HR 3–8s).
3. **Subject-normalized EEG** (z-score within session).
4. **Temporal dynamics**: first differences + rolling slopes, not just raw lags.
5. **CatBoost added** as third model (better for small-N cross-subject).
6. **Lighter regularization** in LGB/XGB to prevent subject overfitting.
7. **Distribution-constrained threshold search** on blended OOF.

# Installing Dependencies

In [1]:
%pip install lightgbm xgboost catboost scikit-learn numpy pandas scipy


[notice] A new release of pip is available: 26.0 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


# Data Load

In [2]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

DATA = '.'

train_labels = pd.read_csv(f'{DATA}/train-label.csv')
test_labels  = pd.read_csv(f'{DATA}/test-label.csv')

trainbvp   = pd.read_csv(f'{DATA}/train-bvp.csv')
traineda   = pd.read_csv(f'{DATA}/train-eda.csv')
traintemp  = pd.read_csv(f'{DATA}/train-temp.csv')
trainhr    = pd.read_csv(f'{DATA}/train-hr.csv')
trainibi   = pd.read_csv(f'{DATA}/train-ibi.csv')
trainbrain = pd.read_csv(f'{DATA}/train-brain.csv')
trainacc   = pd.read_csv(f'{DATA}/train-acc.csv')

testbvp    = pd.read_csv(f'{DATA}/test-bvp.csv')
testeda    = pd.read_csv(f'{DATA}/test-eda.csv')
testtemp   = pd.read_csv(f'{DATA}/test-temp.csv')
testhr     = pd.read_csv(f'{DATA}/test-hr.csv')
testibi    = pd.read_csv(f'{DATA}/test-ibi.csv')
testbrain  = pd.read_csv(f'{DATA}/test-brain.csv')
testacc    = pd.read_csv(f'{DATA}/test-acc.csv')

for df in [train_labels, test_labels,
           trainbvp, traineda, traintemp, trainhr, trainibi, trainbrain, trainacc,
           testbvp,  testeda,  testtemp,  testhr,  testibi,  testbrain,  testacc]:
    df['timestamp'] = pd.to_numeric(df['timestamp'])

for df in [trainacc, testacc]:
    df['magnitude'] = np.sqrt(df['x']**2 + df['y']**2 + df['z']**2)

EEG_COLS = ['delta','theta','lowAlpha','highAlpha',
            'lowBeta','highBeta','lowGamma','middleGamma']
for df in [trainbrain, testbrain]:
    for col in EEG_COLS:
        df[col] = np.log1p(df[col])

print('Data loaded.')
print('Train labels:', train_labels.shape, '| Test labels:', test_labels.shape)

Data loaded.
Train labels: (1456, 4) | Test labels: (1496, 4)


## 1. Subject Baselines + Dead-Sensor Detection + EEG Session Z-Norm

In [3]:
def compute_subject_baseline(sensor_df, val_col):
    out = {}
    for pid, grp in sensor_df.groupby('pid'):
        v = grp[val_col].dropna()
        out[pid] = (v.mean(), v.std() + 1e-8)
    return out

def compute_session_bounds(label_df):
    out = {}
    for pid, grp in label_df.groupby('pid'):
        out[pid] = (grp['timestamp'].min(), grp['timestamp'].max())
    return out

def eda_dead_subjects(eda_df, threshold=0.80):
    dead = set()
    for pid, grp in eda_df.groupby('pid'):
        zr = (grp['value'] == 0).mean()
        if zr > threshold:
            dead.add(pid)
            print(f'  {pid}: EDA zero ratio={zr:.1%} → DEAD')
    return dead

# Session-z-normalize each EEG band within (pid) so cross-subject scales align
def zscore_eeg_within_subject(brain_df):
    df = brain_df.copy()
    for col in EEG_COLS:
        means = df.groupby('pid')[col].transform('mean')
        stds  = df.groupby('pid')[col].transform('std') + 1e-8
        df[f'{col}_z'] = (df[col] - means) / stds
    return df

trainbrain = zscore_eeg_within_subject(trainbrain)
testbrain  = zscore_eeg_within_subject(testbrain)
EEG_Z_COLS = [f'{c}_z' for c in EEG_COLS]

bl_tr_hr   = compute_subject_baseline(trainhr,   'value')
bl_tr_eda  = compute_subject_baseline(traineda,  'value')
bl_tr_temp = compute_subject_baseline(traintemp, 'value')
bl_tr_ibi  = compute_subject_baseline(trainibi,  'value')
bl_tr_acc  = compute_subject_baseline(trainacc,  'magnitude')
bl_tr_bvp  = compute_subject_baseline(trainbvp,  'value')

bl_te_hr   = compute_subject_baseline(testhr,   'value')
bl_te_eda  = compute_subject_baseline(testeda,  'value')
bl_te_temp = compute_subject_baseline(testtemp, 'value')
bl_te_ibi  = compute_subject_baseline(testibi,  'value')
bl_te_acc  = compute_subject_baseline(testacc,  'magnitude')
bl_te_bvp  = compute_subject_baseline(testbvp,  'value')

sess_tr = compute_session_bounds(train_labels)
sess_te = compute_session_bounds(test_labels)

print('Train EDA dead sensors:')
TRAIN_EDA_DEAD = eda_dead_subjects(traineda)
print('Test EDA dead sensors:')
TEST_EDA_DEAD  = eda_dead_subjects(testeda)
print(f'Train dead: {TRAIN_EDA_DEAD} | Test dead: {TEST_EDA_DEAD}')

Train EDA dead sensors:
  70N8: EDA zero ratio=99.6% → DEAD
  Y21H: EDA zero ratio=99.4% → DEAD
Test EDA dead sensors:
Train dead: {'70N8', 'Y21H'} | Test dead: set()


## 2. Feature Extraction — Physiologically-Aware

**Window strategy:**
- Symmetric windows for baseline stats: 2.5s, 5s, 10s
- **Asymmetric EDA window**: [ts-1s, ts+5s] — captures SCR which peaks 1–4s after stimulus
- **Asymmetric HR window**: [ts-2s, ts+8s] — captures cardiac arousal response (3–8s lag)

**New HRV features (from IBI):** pNN50, SDNN, LF/HF proxy via short-time FFT.

**New EDA features:** phasic (high-pass via first difference), tonic (low-pass via rolling mean), SCR-like peak count.

**New BVP features:** peak-detected pulse rate variability as backup HRV channel.

**Temporal dynamics:** first differences and rolling slopes of HR/EDA/TEMP (not just lagged raw values).

In [4]:
from scipy.signal import find_peaks

WINDOWS_MS  = [2500, 5000, 10000]
ROLL_WIN_MS = 30000

# Asymmetric windows (in ms) — (lookback, lookahead)
EDA_ASYM = (1000, 5000)   # SCR peaks 1–4s post-stimulus
HR_ASYM  = (2000, 8000)   # cardiac response 3–8s lag


def win_stats(vals, prefix):
    feat = {}
    n = len(vals)
    if n >= 2:
        feat[f'{prefix}_mean']  = np.mean(vals)
        feat[f'{prefix}_std']   = np.std(vals)
        feat[f'{prefix}_range'] = np.max(vals) - np.min(vals)
        feat[f'{prefix}_slope'] = np.polyfit(np.arange(n), vals, 1)[0]
        feat[f'{prefix}_p25']   = np.percentile(vals, 25)
        feat[f'{prefix}_p75']   = np.percentile(vals, 75)
    else:
        for s in ['mean','std','range','slope','p25','p75']:
            feat[f'{prefix}_{s}'] = np.nan
    return feat


def nan_eda_features(feat, prefix):
    """Null out all EDA-derived features matching prefix (dead sensor protection)."""
    keep_keys = {f'{prefix}_valid', f'{prefix}_zero_ratio'}
    for key in list(feat.keys()):
        if prefix in key and key not in keep_keys:
            feat[key] = np.nan
    return feat


def hrv_features(ibi_v):
    """HRV from inter-beat intervals. Returns dict."""
    out = {}
    if len(ibi_v) < 3:
        for k in ['mean','sdnn','rmssd','pnn50','lfhf_proxy']:
            out[k] = np.nan
        return out
    out['mean']  = np.mean(ibi_v)
    out['sdnn']  = np.std(ibi_v)
    diffs        = np.diff(ibi_v)
    out['rmssd'] = np.sqrt(np.mean(diffs**2))
    # pNN50: % of successive IBI differences > 50ms
    out['pnn50'] = np.mean(np.abs(diffs) > 50.0) if len(diffs) >= 1 else np.nan
    # LF/HF proxy: ratio of variance in low-freq (slow drift) vs high-freq (RMSSD)
    # Cheap surrogate without resampling — uses long-vs-short variance structure.
    if len(ibi_v) >= 6:
        long_var  = np.var(ibi_v - np.convolve(ibi_v, np.ones(3)/3, mode='same'))
        short_var = np.var(diffs)
        out['lfhf_proxy'] = long_var / (short_var + 1e-8)
    else:
        out['lfhf_proxy'] = np.nan
    return out


def eda_phasic_tonic(eda_v):
    """Decompose EDA into phasic (fast, SCR-like) and tonic (slow, level) parts."""
    out = {}
    if len(eda_v) < 3:
        for k in ['phasic_mean','phasic_max','phasic_energy','tonic_mean','tonic_slope','scr_count']:
            out[k] = np.nan
        return out
    # Tonic = rolling mean (low-pass)
    k = max(3, len(eda_v) // 4)
    tonic = np.convolve(eda_v, np.ones(k)/k, mode='same')
    phasic = eda_v - tonic
    out['phasic_mean']   = np.mean(np.abs(phasic))
    out['phasic_max']    = np.max(phasic)
    out['phasic_energy'] = np.mean(phasic**2)
    out['tonic_mean']    = np.mean(tonic)
    out['tonic_slope']   = np.polyfit(np.arange(len(tonic)), tonic, 1)[0]
    # SCR-like peak count: peaks in phasic component above 0.05 uS-ish threshold
    try:
        peaks, _ = find_peaks(phasic, prominence=max(0.01, np.std(phasic)*0.5))
        out['scr_count'] = len(peaks)
    except Exception:
        out['scr_count'] = 0
    return out


def bvp_hrv_features(bvp_v):
    """Pulse-rate variability from BVP via peak detection. Backup HRV channel."""
    out = {}
    if len(bvp_v) < 20:
        for k in ['pulse_n','pulse_sdnn','pulse_rmssd']:
            out[k] = np.nan
        return out
    try:
        # BVP peaks — minimum distance ~50 samples at 64Hz ≈ 0.78s (heart rate < 130bpm)
        peaks, _ = find_peaks(bvp_v, distance=10, prominence=np.std(bvp_v)*0.3)
        if len(peaks) >= 3:
            intervals = np.diff(peaks)
            out['pulse_n']     = len(peaks)
            out['pulse_sdnn']  = np.std(intervals)
            out['pulse_rmssd'] = np.sqrt(np.mean(np.diff(intervals)**2)) if len(intervals) >= 2 else np.nan
        else:
            for k in ['pulse_n','pulse_sdnn','pulse_rmssd']:
                out[k] = np.nan
    except Exception:
        for k in ['pulse_n','pulse_sdnn','pulse_rmssd']:
            out[k] = np.nan
    return out


def extract_all_features(label_df, is_train,
                         hr_df, eda_df, temp_df, ibi_df, acc_df, brain_df, bvp_df,
                         bl_hr, bl_eda, bl_temp, bl_ibi, bl_acc, bl_bvp,
                         sess_bounds, eda_dead_set):

    for df in [hr_df, eda_df, temp_df, ibi_df, acc_df, brain_df, bvp_df]:
        df.sort_values(['pid','timestamp'], inplace=True)

    records = []
    for _, row in label_df.iterrows():
        pid, ts = row['pid'], row['timestamp']
        feat = {'pid': pid, 'timestamp': ts}
        if is_train:
            feat['arousal'] = row['arousal']

        t_min, t_max = sess_bounds.get(pid, (ts, ts))
        feat['session_pos']  = (ts - t_min) / (t_max - t_min + 1e-8)
        feat['pid_eda_dead'] = 1 if pid in eda_dead_set else 0

        feat['bl_hr']   = bl_hr.get(pid,   (np.nan,1))[0]
        feat['bl_eda']  = bl_eda.get(pid,  (np.nan,1))[0]
        feat['bl_temp'] = bl_temp.get(pid, (np.nan,1))[0]
        feat['bl_ibi']  = bl_ibi.get(pid,  (np.nan,1))[0]
        feat['bl_bvp']  = bl_bvp.get(pid,  (np.nan,1))[0]

        def get_sym(df_, col, hw):
            s = df_[df_.pid == pid]
            return s[(s.timestamp >= ts - hw) & (s.timestamp < ts + hw)][col].values

        def get_asym(df_, col, lookback, lookahead):
            s = df_[df_.pid == pid]
            return s[(s.timestamp >= ts - lookback) & (s.timestamp < ts + lookahead)][col].values

        # ---- Asymmetric windows for stimulus-locked responses ----
        # EDA asymmetric (captures SCR)
        eda_a = get_asym(eda_df, 'value', *EDA_ASYM)
        if len(eda_a) >= 3 and pid not in eda_dead_set:
            bl_m, bl_s = bl_eda.get(pid, (np.nan, 1))
            feat['eda_asym_mean']  = np.mean(eda_a)
            feat['eda_asym_dev']   = (np.mean(eda_a) - bl_m) / bl_s
            feat['eda_asym_slope'] = np.polyfit(np.arange(len(eda_a)), eda_a, 1)[0]
            feat['eda_asym_max']   = np.max(eda_a)
            pt = eda_phasic_tonic(eda_a)
            for k,v in pt.items():
                feat[f'eda_asym_{k}'] = v
        else:
            for k in ['mean','dev','slope','max','phasic_mean','phasic_max',
                      'phasic_energy','tonic_mean','tonic_slope','scr_count']:
                feat[f'eda_asym_{k}'] = np.nan

        # HR asymmetric (cardiac response lag)
        hr_a = get_asym(hr_df, 'value', *HR_ASYM)
        if len(hr_a) >= 2:
            bl_m, bl_s = bl_hr.get(pid, (np.nan, 1))
            feat['hr_asym_mean']  = np.mean(hr_a)
            feat['hr_asym_std']   = np.std(hr_a)
            feat['hr_asym_dev']   = (np.mean(hr_a) - bl_m) / bl_s
            feat['hr_asym_slope'] = np.polyfit(np.arange(len(hr_a)), hr_a, 1)[0]
            feat['hr_asym_max']   = np.max(hr_a)
        else:
            for k in ['mean','std','dev','slope','max']:
                feat[f'hr_asym_{k}'] = np.nan

        # ---- Symmetric multi-window features ----
        for hw in WINDOWS_MS:
            wl = f'w{hw//1000}s'

            # HR
            hr_v = get_sym(hr_df, 'value', hw)
            feat.update(win_stats(hr_v, f'hr_{wl}'))
            bl_m, bl_s = bl_hr.get(pid, (np.nan, 1))
            feat[f'hr_{wl}_dev'] = (np.mean(hr_v) - bl_m) / bl_s if len(hr_v) >= 1 else np.nan

            # EDA (symmetric, with NaN masking for dead sensors)
            eda_v = get_sym(eda_df, 'value', hw)
            feat.update(win_stats(eda_v, f'eda_{wl}'))
            bl_m, bl_s = bl_eda.get(pid, (np.nan, 1))
            if len(eda_v) >= 1:
                zr = np.mean(eda_v == 0)
                feat[f'eda_{wl}_zero_ratio'] = zr
                feat[f'eda_{wl}_valid']       = 0 if zr > 0.5 else 1
                feat[f'eda_{wl}_dev']         = (np.mean(eda_v) - bl_m) / bl_s
                nz = eda_v[eda_v != 0]
                feat[f'eda_{wl}_nz_mean'] = np.mean(nz) if len(nz) >= 1 else np.nan
                feat[f'eda_{wl}_nz_frac'] = len(nz) / len(eda_v)
                if zr > 0.5 or pid in eda_dead_set:
                    feat = nan_eda_features(feat, f'eda_{wl}')
            else:
                for k in ['zero_ratio','valid','dev','nz_mean','nz_frac']:
                    feat[f'eda_{wl}_{k}'] = np.nan
                feat = nan_eda_features(feat, f'eda_{wl}')

            # TEMP
            temp_v = get_sym(temp_df, 'value', hw)
            feat.update(win_stats(temp_v, f'temp_{wl}'))
            bl_m, bl_s = bl_temp.get(pid, (np.nan, 1))
            feat[f'temp_{wl}_dev'] = (np.mean(temp_v) - bl_m) / bl_s if len(temp_v) >= 1 else np.nan

            # IBI — richer HRV features
            ibi_v = get_sym(ibi_df, 'value', hw)
            hrv = hrv_features(ibi_v)
            bl_m, bl_s = bl_ibi.get(pid, (np.nan, 1))
            for k, v in hrv.items():
                feat[f'ibi_{wl}_{k}'] = v
            feat[f'ibi_{wl}_dev'] = (hrv['mean'] - bl_m) / bl_s if not np.isnan(hrv['mean']) else np.nan

            # ACC
            acc_v = get_sym(acc_df, 'magnitude', hw)
            bl_m, bl_s = bl_acc.get(pid, (np.nan, 1))
            if len(acc_v) >= 5:
                feat[f'acc_{wl}_mean']   = np.mean(acc_v)
                feat[f'acc_{wl}_std']    = np.std(acc_v)
                feat[f'acc_{wl}_energy'] = np.mean(acc_v**2)
                feat[f'acc_{wl}_dev']    = (np.mean(acc_v) - bl_m) / bl_s
            else:
                for s in ['mean','std','energy','dev']:
                    feat[f'acc_{wl}_{s}'] = np.nan

            # BVP — std/range/iqr AND peak-based HRV
            bvp_v = get_sym(bvp_df, 'value', hw)
            bl_m, bl_s = bl_bvp.get(pid, (np.nan, 1))
            if len(bvp_v) >= 10:
                feat[f'bvp_{wl}_std']   = np.std(bvp_v)
                feat[f'bvp_{wl}_range'] = np.max(bvp_v) - np.min(bvp_v)
                feat[f'bvp_{wl}_iqr']   = np.percentile(bvp_v,75) - np.percentile(bvp_v,25)
                feat[f'bvp_{wl}_dev']   = (np.mean(bvp_v) - bl_m) / bl_s
                bvp_hrv = bvp_hrv_features(bvp_v)
                for k, v in bvp_hrv.items():
                    feat[f'bvp_{wl}_{k}'] = v
            else:
                for s in ['std','range','iqr','dev','pulse_n','pulse_sdnn','pulse_rmssd']:
                    feat[f'bvp_{wl}_{s}'] = np.nan

            # EEG — session-z-normalized + ratios
            s_eeg = brain_df[brain_df.pid == pid]
            win_eeg = s_eeg[(s_eeg.timestamp >= ts-hw) & (s_eeg.timestamp < ts+hw)]
            eps = 1e-8
            if len(win_eeg) >= 1:
                # Z-normalized band means
                for col in EEG_Z_COLS:
                    feat[f'eeg_{col}_{wl}'] = win_eeg[col].mean()
                # Raw-log band means (kept for ratios)
                for col in EEG_COLS:
                    feat[f'eeg_{col}_raw_{wl}'] = win_eeg[col].mean()
                th = feat[f'eeg_theta_raw_{wl}']
                la = feat[f'eeg_lowAlpha_raw_{wl}']
                ha = feat[f'eeg_highAlpha_raw_{wl}']
                lb = feat[f'eeg_lowBeta_raw_{wl}']
                hb = feat[f'eeg_highBeta_raw_{wl}']
                lg = feat[f'eeg_lowGamma_raw_{wl}']
                de = feat[f'eeg_delta_raw_{wl}']
                feat[f'eeg_theta_alpha_{wl}']  = th / (la + ha + eps)
                feat[f'eeg_beta_alpha_{wl}']   = (lb + hb) / (la + ha + eps)
                feat[f'eeg_theta_beta_{wl}']   = th / (lb + hb + eps)  # arousal-relevant
                feat[f'eeg_hbeta_lgamma_{wl}'] = hb / (lg + eps)
                feat[f'eeg_engage_{wl}']        = hb / (de + th + eps)
            else:
                for col in EEG_Z_COLS:
                    feat[f'eeg_{col}_{wl}'] = np.nan
                for col in EEG_COLS:
                    feat[f'eeg_{col}_raw_{wl}'] = np.nan
                for r in ['theta_alpha','beta_alpha','theta_beta','hbeta_lgamma','engage']:
                    feat[f'eeg_{r}_{wl}'] = np.nan

        # ---- Rolling deviations (30s past vs current) ----
        for sensor, df_, col in [('hr', hr_df, 'value'), ('temp', temp_df, 'value')]:
            s = df_[df_.pid == pid]
            past = s[(s.timestamp >= ts - ROLL_WIN_MS) & (s.timestamp < ts)][col].values
            cur  = s[(s.timestamp >= ts - 2500)        & (s.timestamp < ts + 2500)][col].values
            if len(past) >= 2 and len(cur) >= 1:
                feat[f'{sensor}_roll_dev'] = (np.mean(cur) - np.mean(past)) / (np.std(past) + 1e-8)
            else:
                feat[f'{sensor}_roll_dev'] = np.nan

        if pid not in eda_dead_set:
            s = eda_df[eda_df.pid == pid]
            past = s[(s.timestamp >= ts - ROLL_WIN_MS) & (s.timestamp < ts)]['value'].values
            cur  = s[(s.timestamp >= ts - 2500)        & (s.timestamp < ts + 2500)]['value'].values
            if len(past) >= 2 and len(cur) >= 1:
                feat['eda_roll_dev'] = (np.mean(cur) - np.mean(past)) / (np.std(past) + 1e-8)
            else:
                feat['eda_roll_dev'] = np.nan
        else:
            feat['eda_roll_dev'] = np.nan

        # ---- Interactions ----
        hr_m  = feat.get('hr_w5s_mean',  np.nan)
        tmp_m = feat.get('temp_w5s_mean',np.nan)
        hr_d  = feat.get('hr_w5s_dev',   np.nan)
        eda_d = feat.get('eda_w5s_dev',  np.nan)
        feat['hr_temp_product']    = hr_m * tmp_m
        feat['dev_hr_eda_product'] = hr_d * eda_d
        # HRV-arousal interaction: low HRV + high EDA = high arousal signature
        rmssd = feat.get('ibi_w5s_rmssd', np.nan)
        feat['hrv_eda_signature']  = eda_d / (rmssd + 1e-3) if not (np.isnan(eda_d) or np.isnan(rmssd)) else np.nan

        records.append(feat)

    return pd.DataFrame(records)


print('Feature extraction ready.')

Feature extraction ready.


In [5]:
print('Extracting TRAIN features...')
train_feats = extract_all_features(
    train_labels, True,
    trainhr, traineda, traintemp, trainibi, trainacc, trainbrain, trainbvp,
    bl_tr_hr, bl_tr_eda, bl_tr_temp, bl_tr_ibi, bl_tr_acc, bl_tr_bvp,
    sess_tr, TRAIN_EDA_DEAD
)
train_feats.insert(0, 'id', train_labels['id'].values)
print('Train features:', train_feats.shape)

Extracting TRAIN features...
Train features: (1456, 221)


In [6]:
print('Extracting TEST features...')
test_feats = extract_all_features(
    test_labels, False,
    testhr, testeda, testtemp, testibi, testacc, testbrain, testbvp,
    bl_te_hr, bl_te_eda, bl_te_temp, bl_te_ibi, bl_te_acc, bl_te_bvp,
    sess_te, TEST_EDA_DEAD
)
test_feats.insert(0, 'id', test_labels['id'].values)
print('Test features:', test_feats.shape)

Extracting TEST features...
Test features: (1496, 220)


## 3. Temporal Dynamics — Diffs, Slopes, Lags

Instead of just `lag1, lag2, roll3`, also compute first-difference and rolling slope (rate of change). These capture trajectory, not just past values.

In [7]:
LAG_BASE = ['hr_w5s_mean','hr_w5s_dev','hr_roll_dev','hr_asym_mean','hr_asym_dev',
            'eda_w5s_mean','eda_w5s_dev','eda_roll_dev','eda_asym_mean','eda_asym_dev',
            'eda_asym_phasic_mean','eda_asym_scr_count',
            'temp_w5s_mean','temp_w5s_dev','temp_roll_dev',
            'bvp_w5s_std','ibi_w5s_rmssd','ibi_w5s_sdnn']
LAG_COLS = [c for c in LAG_BASE if c in train_feats.columns]


def add_temporal_features(df, cols, lags=(1, 2)):
    df = df.sort_values(['pid','timestamp']).copy()
    # Raw lags
    for lag in lags:
        for col in cols:
            df[f'{col}_lag{lag}'] = df.groupby('pid')[col].shift(lag)
    # First differences (trajectory direction)
    for col in cols:
        df[f'{col}_diff1'] = df.groupby('pid')[col].diff(1)
        df[f'{col}_diff2'] = df.groupby('pid')[col].diff(2)
    # Rolling slope over last 3 timepoints
    for col in cols:
        df[f'{col}_roll3_mean'] = df.groupby('pid')[col].transform(
            lambda x: x.shift(1).rolling(3, min_periods=1).mean()
        )
        df[f'{col}_roll3_std'] = df.groupby('pid')[col].transform(
            lambda x: x.shift(1).rolling(3, min_periods=2).std()
        )
    return df


train_feats = add_temporal_features(train_feats, LAG_COLS)
test_feats  = add_temporal_features(test_feats,  LAG_COLS)

META_COLS = ['id','pid','timestamp','arousal']
FEAT_COLS = [c for c in train_feats.columns if c not in META_COLS]
print(f'Total features: {len(FEAT_COLS)}')

Total features: 325


## 4. Training Setup

In [8]:
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import balanced_accuracy_score, classification_report

train_feats_sorted = train_feats.sort_values(['pid','timestamp']).reset_index(drop=True)
X_all  = train_feats_sorted[FEAT_COLS].values.astype(np.float32)
y_all  = (train_feats_sorted['arousal'].values - 1).astype(int)
pids   = train_feats_sorted['pid'].values
X_test = test_feats[FEAT_COLS].values.astype(np.float32)

cw = compute_class_weight('balanced', classes=np.arange(5), y=y_all)
TRAIN_PRIOR = np.bincount(y_all, minlength=5) / len(y_all)
print('Class weights:', {f'A{i+1}': round(w,2) for i,w in enumerate(cw)})
print('Training prior:', {f'A{i+1}': round(p,3) for i,p in enumerate(TRAIN_PRIOR)})

Class weights: {'A1': np.float64(5.29), 'A2': np.float64(0.68), 'A3': np.float64(0.53), 'A4': np.float64(0.84), 'A5': np.float64(4.04)}
Training prior: {'A1': np.float64(0.038), 'A2': np.float64(0.295), 'A3': np.float64(0.38), 'A4': np.float64(0.237), 'A5': np.float64(0.049)}


## 5. LOSO CV — LGB + XGB + CatBoost

**Regularization is lighter than v18** to prevent subject overfitting:
- LGB: `num_leaves 63 → 31`, `min_child_samples 15 → 30`, lambdas `0.3 → 0.5`
- XGB: `max_depth 5 → 4`, `min_child_weight 10 → 20`
- CatBoost: depth 6, l2_leaf_reg 5 (CatBoost's symmetric trees regularize naturally)

In [9]:
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier, Pool

TRAIN_PIDS = sorted(train_feats_sorted['pid'].unique())
SEEDS      = [42, 7, 123]

oof_lgb  = np.zeros((len(train_feats_sorted), 5), dtype=np.float64)
oof_xgb  = np.zeros((len(train_feats_sorted), 5), dtype=np.float64)
oof_cat  = np.zeros((len(train_feats_sorted), 5), dtype=np.float64)
test_lgb = np.zeros((len(test_feats), 5), dtype=np.float64)
test_xgb = np.zeros((len(test_feats), 5), dtype=np.float64)
test_cat = np.zeros((len(test_feats), 5), dtype=np.float64)
loso_lgb, loso_xgb, loso_cat = [], [], []


def get_lgb_params(seed):
    return dict(
        objective='multiclass', num_class=5, metric='multi_logloss',
        num_leaves=31, learning_rate=0.03,
        feature_fraction=0.6, bagging_fraction=0.8, bagging_freq=5,
        min_child_samples=30, lambda_l1=0.5, lambda_l2=0.5,
        max_depth=6, verbose=-1, seed=seed, n_jobs=-1
    )


def get_xgb_params(seed):
    return dict(
        objective='multi:softprob', num_class=5, eval_metric='mlogloss',
        max_depth=4, learning_rate=0.03,
        subsample=0.8, colsample_bytree=0.6,
        min_child_weight=20, reg_alpha=0.5, reg_lambda=0.5,
        seed=seed, verbosity=0, nthread=-1
    )


def get_cat_params(seed):
    return dict(
        loss_function='MultiClass', classes_count=5,
        eval_metric='MultiClass',
        iterations=1500, learning_rate=0.03,
        depth=6, l2_leaf_reg=5.0,
        random_strength=1.0, bagging_temperature=0.5,
        random_seed=seed, verbose=False,
        thread_count=-1, allow_writing_files=False,
    )


for fold_pid in TRAIN_PIDS:
    tr_mask = pids != fold_pid
    va_mask = pids == fold_pid
    X_tr, y_tr = X_all[tr_mask], y_all[tr_mask]
    X_va, y_va = X_all[va_mask], y_all[va_mask]
    sw_tr = cw[y_tr]

    f_lgb = np.zeros((va_mask.sum(), 5))
    f_xgb = np.zeros((va_mask.sum(), 5))
    f_cat = np.zeros((va_mask.sum(), 5))
    t_lgb = np.zeros((len(test_feats), 5))
    t_xgb = np.zeros((len(test_feats), 5))
    t_cat = np.zeros((len(test_feats), 5))

    for seed in SEEDS:
        # LGB
        dtr = lgb.Dataset(X_tr, label=y_tr, weight=sw_tr)
        dva = lgb.Dataset(X_va, label=y_va, reference=dtr)
        m = lgb.train(
            get_lgb_params(seed), dtr, num_boost_round=2000,
            valid_sets=[dva],
            callbacks=[lgb.early_stopping(150, verbose=False),
                       lgb.log_evaluation(period=-1)],
        )
        f_lgb += m.predict(X_va)   / len(SEEDS)
        t_lgb += m.predict(X_test) / len(SEEDS)

        # XGB
        dtr_x = xgb.DMatrix(X_tr, label=y_tr, weight=sw_tr)
        dva_x = xgb.DMatrix(X_va, label=y_va)
        m_x = xgb.train(
            get_xgb_params(seed), dtr_x, num_boost_round=2000,
            evals=[(dva_x, 'val')],
            early_stopping_rounds=150, verbose_eval=False,
        )
        f_xgb += m_x.predict(dva_x).reshape(-1,5)               / len(SEEDS)
        t_xgb += m_x.predict(xgb.DMatrix(X_test)).reshape(-1,5) / len(SEEDS)

        # CatBoost
        train_pool = Pool(X_tr, label=y_tr, weight=sw_tr)
        val_pool   = Pool(X_va, label=y_va)
        m_c = CatBoostClassifier(**get_cat_params(seed),
                                  early_stopping_rounds=150)
        m_c.fit(train_pool, eval_set=val_pool, verbose=False)
        f_cat += m_c.predict_proba(X_va)   / len(SEEDS)
        t_cat += m_c.predict_proba(X_test) / len(SEEDS)

    oof_lgb[va_mask] = f_lgb
    oof_xgb[va_mask] = f_xgb
    oof_cat[va_mask] = f_cat
    test_lgb += t_lgb / len(TRAIN_PIDS)
    test_xgb += t_xgb / len(TRAIN_PIDS)
    test_cat += t_cat / len(TRAIN_PIDS)

    ba_l = balanced_accuracy_score(y_va, f_lgb.argmax(axis=1))
    ba_x = balanced_accuracy_score(y_va, f_xgb.argmax(axis=1))
    ba_c = balanced_accuracy_score(y_va, f_cat.argmax(axis=1))
    loso_lgb.append(ba_l)
    loso_xgb.append(ba_x)
    loso_cat.append(ba_c)
    print(f'  {fold_pid} — LGB: {ba_l:.4f} | XGB: {ba_x:.4f} | CAT: {ba_c:.4f}')

print(f'\nLOSO LGB mean: {np.mean(loso_lgb):.4f} ± {np.std(loso_lgb):.4f}')
print(f'LOSO XGB mean: {np.mean(loso_xgb):.4f} ± {np.std(loso_xgb):.4f}')
print(f'LOSO CAT mean: {np.mean(loso_cat):.4f} ± {np.std(loso_cat):.4f}')

  01Z2 — LGB: 0.2444 | XGB: 0.0389 | CAT: 0.4889
  70N8 — LGB: 0.1617 | XGB: 0.1762 | CAT: 0.2287
  7PF3 — LGB: 0.2049 | XGB: 0.1796 | CAT: 0.1887
  CQ2G — LGB: 0.2812 | XGB: 0.2532 | CAT: 0.1350
  D1XP — LGB: 0.2294 | XGB: 0.2631 | CAT: 0.2500
  DT5C — LGB: 0.2528 | XGB: 0.2274 | CAT: 0.2143
  F1ZM — LGB: 0.1615 | XGB: 0.1418 | CAT: 0.0576
  LIUY — LGB: 0.4925 | XGB: 0.5415 | CAT: 0.4877
  SE4Q — LGB: 0.4841 | XGB: 0.4710 | CAT: 0.4948
  TPQI — LGB: 0.0781 | XGB: 0.1072 | CAT: 0.1086
  Y21H — LGB: 0.1024 | XGB: 0.1314 | CAT: 0.2148

LOSO LGB mean: 0.2448 ± 0.1292
LOSO XGB mean: 0.2301 ± 0.1450
LOSO CAT mean: 0.2608 ± 0.1505


## 6. Three-Way Blend Search (coarse grid)

In [10]:
best_ba, best_w = 0.0, (1/3, 1/3, 1/3)
for wl in np.arange(0.0, 1.01, 0.1):
    for wx in np.arange(0.0, 1.01 - wl, 0.1):
        wc = 1.0 - wl - wx
        if wc < -1e-6:
            continue
        wc = max(wc, 0.0)
        blend = wl * oof_lgb + wx * oof_xgb + wc * oof_cat
        ba    = balanced_accuracy_score(y_all, blend.argmax(axis=1))
        if ba > best_ba:
            best_ba, best_w = ba, (wl, wx, wc)

wl, wx, wc = best_w
print(f'Best weights — LGB: {wl:.2f} | XGB: {wx:.2f} | CAT: {wc:.2f}')
print(f'Argmax OOF BA: {best_ba:.4f}')

oof_blend  = wl * oof_lgb  + wx * oof_xgb  + wc * oof_cat
test_blend = wl * test_lgb + wx * test_xgb + wc * test_cat

Best weights — LGB: 0.10 | XGB: 0.00 | CAT: 0.90
Argmax OOF BA: 0.2048


## 7. Distribution-Constrained Threshold Search

Argmax on raw probabilities is biased by the prior. Instead: find a per-class additive bias that maximizes OOF BA while keeping the predicted test distribution within reasonable bounds of the training prior. This is consistent with the strategy that recovered points in your earlier competitions.

In [11]:
from itertools import product

def apply_bias(probs, bias):
    """Add per-class bias in log space, then argmax."""
    return (np.log(probs + 1e-12) + bias).argmax(axis=1)

# Coarse-to-fine search
BIAS_RANGE_COARSE = np.arange(-1.0, 1.01, 0.25)
BIAS_RANGE_FINE   = np.arange(-0.5, 0.51, 0.1)

# Constraint: predicted distribution on TEST must stay within +/- 50% of training prior
def dist_constraint_ok(preds, prior, slack=0.5):
    dist = np.bincount(preds, minlength=5) / len(preds)
    for i in range(5):
        lo = max(0.01, prior[i] * (1 - slack))
        hi = min(0.99, prior[i] * (1 + slack))
        if dist[i] < lo - 0.05 or dist[i] > hi + 0.05:
            return False
    return True

best_score = -1.0
best_bias  = np.zeros(5)

# Coarse pass
for b in product(BIAS_RANGE_COARSE, repeat=5):
    bias = np.array(b)
    oof_pred  = apply_bias(oof_blend,  bias)
    test_pred = apply_bias(test_blend, bias)
    if not dist_constraint_ok(test_pred, TRAIN_PRIOR, slack=0.6):
        continue
    ba = balanced_accuracy_score(y_all, oof_pred)
    if ba > best_score:
        best_score = ba
        best_bias  = bias.copy()

print(f'Coarse best — OOF BA: {best_score:.4f} | bias: {best_bias}')

# Fine pass around the coarse optimum
for b in product(BIAS_RANGE_FINE, repeat=5):
    bias = best_bias + np.array(b)
    oof_pred  = apply_bias(oof_blend,  bias)
    test_pred = apply_bias(test_blend, bias)
    if not dist_constraint_ok(test_pred, TRAIN_PRIOR, slack=0.6):
        continue
    ba = balanced_accuracy_score(y_all, oof_pred)
    if ba > best_score:
        best_score = ba
        best_bias  = bias.copy()

print(f'Fine best — OOF BA: {best_score:.4f} | bias: {best_bias}')

FINAL_BIAS = best_bias
oof_pred  = apply_bias(oof_blend,  FINAL_BIAS)
test_pred = apply_bias(test_blend, FINAL_BIAS) + 1

print(f'\nFinal OOF BA: {balanced_accuracy_score(y_all, oof_pred):.4f}')
print('Test distribution:')
print(pd.Series(test_pred).value_counts().sort_index())
print('Train prior counts:', {f'A{i+1}': int(p*len(test_pred)) for i,p in enumerate(TRAIN_PRIOR)})

Coarse best — OOF BA: 0.3168 | bias: [-1.    0.    0.25  0.5   0.75]
Fine best — OOF BA: 0.3168 | bias: [-1.    0.    0.25  0.5   0.75]

Final OOF BA: 0.3168
Test distribution:
1      4
2    172
3    654
4    474
5    192
Name: count, dtype: int64
Train prior counts: {'A1': 56, 'A2': 441, 'A3': 569, 'A4': 354, 'A5': 73}


## 8. OOF Diagnosis

In [12]:
print('=== OOF Classification Report (after bias) ===')
print(classification_report(y_all, oof_pred,
                            target_names=[f'Arousal {i+1}' for i in range(5)]))

print('Per-subject LOSO BA:')
for pid, ba_l, ba_x, ba_c in zip(TRAIN_PIDS, loso_lgb, loso_xgb, loso_cat):
    flag = ' ← LOW' if max(ba_l, ba_x, ba_c) < 0.25 else ''
    print(f'  {pid}: LGB={ba_l:.4f} XGB={ba_x:.4f} CAT={ba_c:.4f}{flag}')

print(f'\nFinal OOF BA (biased): {balanced_accuracy_score(y_all, oof_pred):.4f}')
print(f'LOSO LGB mean        : {np.mean(loso_lgb):.4f}')
print(f'LOSO XGB mean        : {np.mean(loso_xgb):.4f}')
print(f'LOSO CAT mean        : {np.mean(loso_cat):.4f}')

=== OOF Classification Report (after bias) ===
              precision    recall  f1-score   support

   Arousal 1       0.12      0.02      0.03        55
   Arousal 2       0.30      0.22      0.25       430
   Arousal 3       0.27      0.21      0.24       554
   Arousal 4       0.52      0.41      0.46       345
   Arousal 5       0.12      0.72      0.21        72

    accuracy                           0.28      1456
   macro avg       0.27      0.32      0.24      1456
weighted avg       0.33      0.28      0.29      1456

Per-subject LOSO BA:
  01Z2: LGB=0.2444 XGB=0.0389 CAT=0.4889
  70N8: LGB=0.1617 XGB=0.1762 CAT=0.2287 ← LOW
  7PF3: LGB=0.2049 XGB=0.1796 CAT=0.1887 ← LOW
  CQ2G: LGB=0.2812 XGB=0.2532 CAT=0.1350
  D1XP: LGB=0.2294 XGB=0.2631 CAT=0.2500
  DT5C: LGB=0.2528 XGB=0.2274 CAT=0.2143
  F1ZM: LGB=0.1615 XGB=0.1418 CAT=0.0576 ← LOW
  LIUY: LGB=0.4925 XGB=0.5415 CAT=0.4877
  SE4Q: LGB=0.4841 XGB=0.4710 CAT=0.4948
  TPQI: LGB=0.0781 XGB=0.1072 CAT=0.1086 ← LOW
  Y21H: L

# Generating Final Submission

In [13]:
submission = pd.DataFrame({
    'id':      test_feats['id'].values,
    'arousal': test_pred,
})

submission.to_csv('submission-v26.csv', index=False)
print('submission-v26.csv saved.')
print(f'Shape: {submission.shape}')
print(submission['arousal'].value_counts().sort_index())

submission-v26.csv saved.
Shape: (1496, 2)
arousal
1      4
2    172
3    654
4    474
5    192
Name: count, dtype: int64
